#Chile - Universidad Adolfo Ibáñez (UAI)
##Curso NLP
### Text Summarisation

# Text Summarisation (Pipeline)

In [1]:
# Instalar Librerías
!pip install -U transformers datasets evaluate --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 121.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.6/503.6 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 58.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.
pylibcudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.


In [2]:
#Importar Librerías
from transformers import pipeline

In [3]:
#Cargar el pipeline de QA (Español)
summarisation_pipeline = pipeline("summarization",
                                  model="facebook/bart-large-cnn")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


In [4]:
#Texto Original
text = """
The World Health Organization (WHO) released a new report on global health trends in 2023.
According to the report, non-communicable diseases such as diabetes and cardiovascular conditions continue to rise,
particularly in low- and middle-income countries. Meanwhile, mental health has become an urgent priority for global public health,
with significant gaps in access to care and resources. The WHO recommends integrated policy frameworks that address both physical
and mental health, alongside new digital health technologies to enhance accessibility and early diagnosis.
"""

In [5]:
#Summarisation
summary = summarisation_pipeline(text,
                                 max_length=60,
                                 min_length=30,
                                 do_sample=False)

In [6]:
#Revisar Resumen
print("📝 Texto Original:\n", text)
print("\n📄 Resumen:\n", summary[0]["summary_text"])

📝 Texto Original:
 
The World Health Organization (WHO) released a new report on global health trends in 2023.
According to the report, non-communicable diseases such as diabetes and cardiovascular conditions continue to rise,
particularly in low- and middle-income countries. Meanwhile, mental health has become an urgent priority for global public health,
with significant gaps in access to care and resources. The WHO recommends integrated policy frameworks that address both physical
and mental health, alongside new digital health technologies to enhance accessibility and early diagnosis.


📄 Resumen:
 The World Health Organization (WHO) released a new report on global health trends in 2023. Non-communicable diseases such as diabetes and cardiovascular conditions continue to rise. Meanwhile, mental health has become an urgent priority for global public health.


# Fine-Tuning Modelo QA (Inglés)

In [7]:
#Importar Librerías
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer
import evaluate
import numpy as np
import torch

In [8]:
#Cargar Dataset
dataset = load_dataset("cnn_dailymail", "3.0.0", split="train[:1%]").train_test_split(test_size=0.2)

README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

In [9]:
#Revisar Ejemplo (Texto Original)
print("📰 Texto original:\n", dataset["train"][0]["article"])

📰 Texto original:
 LAGOS, Nigeria (Reuters) -- Nigeria's television survival show has been suspended after a contestant drowned in preparation for the program, said Dutch brewer Heineken's local unit which is sponsoring the show. Anthony Ogadje, 25, and nine other contestants had gone to Shere Hills Lake in Nigeria's hilly Plateau State to prepare for the "Gulder Ultimate Search," which sets a variety of physical challenges for participants. A statement from Nigerian Breweries on Monday said Ogadje died suddenly and he was thought to have drowned. "All attempts to revive him by the attendant medical team and the lifeguards, including his fellow contestants, failed," said Nigerian Breweries, which is majority-owned by the Dutch giant. Broadcasting had been due to start on Thursday. In the show, the weakest contestants are evicted one by one until a winner emerges. The prize money is a big attraction in a country where most people live in extreme poverty and benefit little from Nigeria's

In [10]:
#Revisar Ejemplo (Resumen)
print("\n📄 Resumen de referencia:\n", dataset["train"][0]["highlights"])


📄 Resumen de referencia:
 Anthony Ogadje, 25, reportedly drowned in Shere Hills Lake .
He was preparing for the show, "Gulder Ultimate Search"
Dutch brewer Heineken's local unit sponsors the program .


In [11]:
#Cargar Tokenizador & Modelo
checkpoint = "facebook/bart-base"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint).to("cuda" if torch.cuda.is_available() else "cpu")

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

In [12]:
#Función Pre-Procesamiento
def preprocess(example):
    model_inputs = tokenizer(example["article"], max_length=512, truncation=True)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(example["highlights"], max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [13]:
#Tokenización & Pre-Procesamiento
tokenized_dataset = dataset.map(preprocess, batched=True)

Map:   0%|          | 0/2296 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4007: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/575 [00:00<?, ? examples/s]

In [14]:
#Configurar Entrenamiento
training_args = TrainingArguments(
    output_dir="./summarisation_finetuned",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=2,
    logging_steps=100,
    report_to="none"
)

In [15]:
#Crear Data Collator
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

In [16]:
#Instalar & Importar Librería
!pip install rouge_score --quiet
import evaluate
import numpy as np

  Preparing metadata (setup.py) ... done


In [17]:
#Crear Set Evaluación (Pequeño)
eval_set = tokenized_dataset["test"].select(range(50))
eval_set = eval_set.map(preprocess, batched=True)

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4007: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


In [18]:
#Configurar Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=eval_set,
    data_collator=data_collator
)

In [19]:
#Fine-Tuning Modelo
trainer.train()

Step,Training Loss
100,2.786900
200,2.400100


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:4037: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=287, training_loss=2.5290075960059615, metrics={'train_runtime': 81.9498, 'train_samples_per_second': 28.017, 'train_steps_per_second': 3.502, 'total_flos': 699977454059520.0, 'train_loss': 2.5290075960059615, 'epoch': 1.0})

In [20]:
#Preparar Inputs (Nuevo Ejemplo)
sample_text = dataset["test"][0]["article"]
inputs = tokenizer(sample_text, return_tensors="pt", truncation=True, max_length=512).to(model.device)

In [21]:
#Generar Resumen (Nuevo Ejemplo)
summary_ids = model.generate(**inputs, max_length=100)
print("📰 Texto original:\n", sample_text)
print("\n📄 Resumen generado:\n", tokenizer.decode(summary_ids[0], skip_special_tokens=True))

📰 Texto original:
 (CNN) -- In a television interview, the mother of a man charged in the murder of an Auburn University freshman repeatedly says she's sorry about the suffering the victim's family is enduring. Courtney Larrell Lockhart was arrested Friday in Phenix City, Alabama, about 35 miles from Auburn. "I never thought Courtney would do this. I never, never thought," Courtney Larrell Lockhart's mother Catherine Williams told CNN affiliate WRBL on Saturday. "But I'm sorry for that family and I'm sorry. I'm just sorry," she said. "I got nothing else to say. I'm just sorry for the loss of that family." Police announced Saturday that they had arrested Lockhart, 23, of Smiths, Alabama, in connection with the shooting death of Lauren Burk, 18, of Marietta, Georgia.  Watch the mother cry and apologize » . Lockhart faces charges of capital murder during a kidnapping, capital murder during a robbery and capital murder during an attempted rape, police said. Also, Lockhart is facing robbery